### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

1. Tracking agent behavior with logging, analytic and debugging
2. Transforming prompts, tool selection and output formatting
3. Adding retries, fallbacks and early termination logic
4. Applying rate limits guradrails and Pll detection

In [4]:
import os
from dotenv import  load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

### Summarization Middleware

Automatically summarize conversation history when approaching token limits preserving recent messages while compressing older context. Summarization is useful for the following:

1. Long-running conversation that exceed context windows
2. Multi-turn dialogues with extensive history
3. Application where preserving full conversation context matters


In [5]:
from langchain.agents import  create_agent
from langchain.agents.middleware import  SummarizationMiddleware

from langgraph.checkpoint.memory import  InMemorySaver
from langchain_core.messages import  HumanMessage, SystemMessage

# Messagebased summarization

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-120b",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [6]:
# Run with thread ID
config = {"configurable":{"thread_id","test-1"}}
print(type(config["configurable"]))   # <class 'dict'> aana chahiye

<class 'set'>


In [7]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "test-1"}}

questions = [
    "what is 2+2?",
    "what is 43/8?",
    "what is 23-324?",
    "what is 878*6?",
]

for q in questions:
    response = agent.invoke({"messages": [HumanMessage(content=q)]}, config)
    print(f"Messages: {response}")
    print(f" Messages: {len(response['messages'])}")
 

Messages: {'messages': [HumanMessage(content='what is 2+2?', additional_kwargs={}, response_metadata={}, id='759aed8a-9d1f-453e-92a1-e1fba7a1617a'), AIMessage(content='2\u202f+\u202f2\u202f=\u202f4.', additional_kwargs={'reasoning_content': 'The user asks a simple math question: "what is 2+2?" Answer: 4.'}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 78, 'total_tokens': 119, 'completion_time': 0.08550913, 'completion_tokens_details': {'reasoning_tokens': 22}, 'prompt_time': 0.004335663, 'prompt_tokens_details': None, 'queue_time': 0.301098805, 'total_time': 0.089844793}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_f640395b96', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a15d0f-f818-7a21-ba27-389956b1a806-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 41, 'total_tokens': 119, 'output_token_details': {'reasoning': 22}})]}

### Token Size

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city:str):
    """ Search hotels return long response to use a tokens""" 
    return f"""Hotels in {city}:
    1. Grand Hotel 5 star, $350/night,spa,pool,gym
    2. city Inn 4 star, $100/night, business center
    3. Budget stay 3 star , $75/night , free WIFI"""  


agent=create_agent(
    model="groq:openai/gpt-oss-120b",
    tools = [search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
             model="groq:openai/gpt-oss-120b",
             trigger=("tokens",550),
             keep=("tokens",200),
        )
    ]

)

config = {"configurable":{"thread_id":"test-1"}}

# Token Counter (Approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 # 4 chars = 1 token

In [9]:
# Run Test 
cities = ["Paris","London","Tokyo","New York","Dubai","Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages":[HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokenss, {len(response["messages"])} messages")
    print(f"{(response["messages"])}")



Paris: ~275 tokenss, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='68b74675-4e82-4ac0-ac5d-302effad7d45'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to find hotels in Paris. Use the function search_hotels with city "Paris".', 'tool_calls': [{'id': 'fc_44fbfbac-11d3-4411-86af-74cf0498b573', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 128, 'total_tokens': 177, 'completion_time': 0.102758704, 'completion_tokens_details': {'reasoning_tokens': 21}, 'prompt_time': 0.005954416, 'prompt_tokens_details': None, 'queue_time': 0.374685758, 'total_time': 0.10871312}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_27194a498a', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a15d10-0229-7a51-859d

### Fraction

In [10]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware

from langchain_core.tools import  tool
from langchain_core.messages import HumanMessage

from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city:str) ->str:
    """ Search Hotels."""
    return f"Hotels in {city}: Grand Hotel $350,city Inn $180, Budget stay $75"

# Low Fraction For Testing

agent=create_agent(
    model="groq:openai/gpt-oss-120b",
    tools = [search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
             model="groq:openai/gpt-oss-120b",
             trigger=("fraction",0.005),# 0.5% = ~640 tokens
             keep=("fraction", 0.002), #0.2% = ~256 tokens

        )
    ]

)

config= {"configurable":{"thread_id":"test-1"}}

# Token Counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) //4

# Test
cities = ["Paris","London","Tokyo","New York","Dubai","Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages":[HumanMessage(content=f" Hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  
    print(f"{city}: ~{tokens} tokens, ({fraction:.4%}), {len(response["messages"])} msgs")
    print(f"{(response["messages"])}")



Paris: ~163 tokens, (0.1273%), 4 msgs
[HumanMessage(content=' Hotels in Paris', additional_kwargs={}, response_metadata={}, id='2b32679f-8381-47b7-87ab-27cf91dcbb4e'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to get hotels in Paris. Use function search_hotels with city "Paris".', 'tool_calls': [{'id': 'fc_6d0e28cf-cba8-408a-9c0e-699910b1e4e2', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 120, 'total_tokens': 167, 'completion_time': 0.097741565, 'completion_tokens_details': {'reasoning_tokens': 19}, 'prompt_time': 0.005132928, 'prompt_tokens_details': None, 'queue_time': 0.332296003, 'total_time': 0.102874493}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_87b3c396db', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a15d11-a5cc-7de1-8dab-f5937d80

### Human In the Loop Middeware

Pause agent execution for human approval, editing or rejection of tool calls before they execute. Human in the loop is useful for the following:
1. High stakes operations requiring human approval for example Database writes, financial transactions
2. Compliance workflows where human oversight is mandatory
3. Long running conversations where human feedback guides the agents


In [11]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware

from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id:str) ->str:
    """Mock function to read an email by its ID""" 
    return f"Email content for ID:{email_id}"

def send_email_tool(recipient:str, subject:str,body:str) ->str:
    """Mock function to send an email"""
    return f"Email sent to {recipient} with subject '{subject}'"

In [12]:
agent= create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]

                },
                "read_email_tool":False,

            }
        )
    ]
)

In [13]:
config= {"configurable": {"thread_id":"test_approve"}}
# step1 : Request
result = agent.invoke(
    {"messages":[HumanMessage(content="Send email to john@test.com with subject 'Hello John' and body 'How are you?")]},
    config=config
)

In [14]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello John' and body 'How are you?", additional_kwargs={}, response_metadata={}, id='30f403c7-cb1a-459a-b117-352c967d0acb'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email. Use send_email_tool.', 'tool_calls': [{'id': 'fc_7735ccdd-be1b-4e5b-a19c-8d4324e7830a', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello John"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 175, 'total_tokens': 234, 'completion_time': 0.124054019, 'completion_tokens_details': {'reasoning_tokens': 12}, 'prompt_time': 0.007049701, 'prompt_tokens_details': None, 'queue_time': 0.302238682, 'total_time': 0.13110372}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8ea50c0161', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_pr

In [15]:
# Step 2 Approval
from langgraph.types import Command
if "__interrupt__" in result:
    print("Paused! Approving....")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"approve"}
                ]
            }
        ),
        config=config
    )

    print(f"Result:{result['messages'][-1].content}")

Paused! Approving....
Result:The email has been sent to **john@test.com** with the subject **“Hello John”** and the body **“How are you?”**. Let me know if there’s anything else you’d like to do!


In [16]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello John' and body 'How are you?", additional_kwargs={}, response_metadata={}, id='30f403c7-cb1a-459a-b117-352c967d0acb'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email. Use send_email_tool.', 'tool_calls': [{'id': 'fc_7735ccdd-be1b-4e5b-a19c-8d4324e7830a', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello John"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 175, 'total_tokens': 234, 'completion_time': 0.124054019, 'completion_tokens_details': {'reasoning_tokens': 12}, 'prompt_time': 0.007049701, 'prompt_tokens_details': None, 'queue_time': 0.302238682, 'total_time': 0.13110372}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8ea50c0161', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_pr

### Reject 


In [17]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware

from langchain_core.messages import HumanMessage

from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id:str) ->str:
    """Mock function to read an email by its ID""" 
    return f"Email content for ID:{email_id}"

def send_email_tool(recipient:str, subject:str,body:str) ->str:
    """Mock function to send an email"""
    return f"Email sent to {recipient} with subject '{subject}'"

agent= create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]

                },
                "read_email_tool":False,

            }
        )
    ]
)


In [18]:
config = {"configurable":{"thread_id":"test-reject-2"}}
# Step 1 Request
result = agent.invoke(
    {"messages":[HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config)

In [19]:
# Step 2 Reject
from langgraph.types import Command
if "__interrupt__" in result:
    print("Paused! Approving....")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type":"reject"}
                ]
            }
        ),
        config=config
    )

    print(f"Result: {result['messages'][-1].content}")

Paused! Approving....
Result: I’m sorry—I wasn’t able to send the email you requested. Would you like me to try sending it again?


In [20]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='5f226306-8fd2-498e-9995-a6a26e8c0e39'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to send an email. Use send_email_tool.', 'tool_calls': [{'id': 'fc_f36e0ba2-2b79-4915-bc01-d194e33fa495', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 174, 'total_tokens': 234, 'completion_time': 0.125561423, 'completion_tokens_details': {'reasoning_tokens': 14}, 'prompt_time': 0.007920134, 'prompt_tokens_details': None, 'queue_time': 0.300660715, 'total_time': 0.133481557}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_dee443f41b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_

### Editing


In [37]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware

from langchain_core.messages import HumanMessage

from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id:str) ->str:
    """Mock function to read an email by its ID""" 
    return f"Email content for ID:{email_id}"

def send_email_tool(recipient:str, subject:str,body:str) ->str:
    """Mock function to send an email"""
    return f"Email sent to {recipient} with subject '{subject}'"

agent= create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]

                },
                "read_email_tool":False,

            }
        )
    ]
)


In [38]:
config ={"configurable":{"thread_id":"test_edit"}}

# step 1 Request with wrong info
result = agent.invoke(
    {"messages":[HumanMessage(content="Send email to wrong@gmail.com with subject 'Test' and body'Hello'")]},
    config=config
)

In [31]:
result

{'messages': [HumanMessage(content="Send email to wrong@gmail.com with subject 'Test' and body'Hello'", additional_kwargs={}, response_metadata={}, id='b72099e7-6e14-4381-a158-9f8e424a2669'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email. Use send_email_tool.', 'tool_calls': [{'id': 'fc_11241379-6984-4f51-88db-3f5f10e8dc25', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@gmail.com","subject":"Test"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 172, 'total_tokens': 228, 'completion_time': 0.117549241, 'completion_tokens_details': {'reasoning_tokens': 12}, 'prompt_time': 0.007463203, 'prompt_tokens_details': None, 'queue_time': 0.374684835, 'total_time': 0.125012444}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c15aa9c1b7', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id=

In [35]:
print("Interrupt details:")
print(result.get("__interrupt__"))

print("\nMessages:")
for message in result.get("messages", []):
    print(message)

Interrupt details:
[Interrupt(value={'action_requests': [{'name': 'send_email_tool', 'args': {'body': 'Hello', 'recipient': 'wrong@gmail.com', 'subject': 'Test'}, 'description': "Tool execution requires approval\n\nTool: send_email_tool\nArgs: {'body': 'Hello', 'recipient': 'wrong@gmail.com', 'subject': 'Test'}"}], 'review_configs': [{'action_name': 'send_email_tool', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='01ff404c603960af40efd86872ac98a0', response_schema=None)]

Messages:
content="Send email to wrong@gmail.com with subject 'Test' and body'Hello'" additional_kwargs={} response_metadata={} id='b72099e7-6e14-4381-a158-9f8e424a2669'
content='' additional_kwargs={'reasoning_content': 'We need to send email. Use send_email_tool.', 'tool_calls': [{'id': 'fc_11241379-6984-4f51-88db-3f5f10e8dc25', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@gmail.com","subject":"Test"}', 'name': 'send_email_tool'}, 'type': 'function'}]} response_metadata={'token_usage': {

In [41]:
from langgraph.types import Command
if "__interrupt__"in result:
    print("Paused! Editing .....")

    result = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "edit",
                    "edited_action": {
                        "name": "send_email_tool",
                        "args": {
                            "recipient": "correct@gmail.com",
                            "subject": "Corrected Subject",
                            "body": "This was edited by human before sending"
                        }
                    }
                }
            ]
        }
    ),
    config=config
)






In [42]:
result

{'messages': [HumanMessage(content="Send email to wrong@gmail.com with subject 'Test' and body'Hello'", additional_kwargs={}, response_metadata={}, id='f90134c5-cb01-484e-8342-35227cc819d9'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "We need to send an email using send_email_tool. The user wants to send email to wrong@gmail.com with subject 'Test' and body 'Hello'. Use the function.", 'tool_calls': [{'id': 'fc_d233be89-6b6f-4a59-95c8-71d24cf5a233', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@gmail.com","subject":"Test"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 80, 'prompt_tokens': 172, 'total_tokens': 252, 'completion_time': 0.16839974, 'completion_tokens_details': {'reasoning_tokens': 36}, 'prompt_time': 0.008375111, 'prompt_tokens_details': None, 'queue_time': 0.300733475, 'total_time': 0.176774851}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_f640395b96', '